# Skill 32: Multi-Head Attention

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Project, split, attend, merge and re-project with H heads, tracking every tensor shape.

## Objectives
- Project embeddings into Q, K and V.
- Split the embedding dimension across heads and transpose sequence/head axes.
- Run attention per head, concatenate heads and project back to `embed_dim`.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$\mathrm{MHA}(X)=[\mathrm{head}_1\Vert\cdots\Vert\mathrm{head}_H]\,W_O,\qquad E=H\,d_h$$
Shapes: $(B,T,E)\to(B,T,H,d_h)\to(B,H,T,d_h)$, attention, then back to $(B,T,E)$. Parameters: $4E^2+4E$.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def split_heads(self, x):
        B, T, E = x.shape
        return x.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, mask=None):
        q = self.split_heads(self.q_proj(x))
        k = self.split_heads(self.k_proj(x))
        v = self.split_heads(self.v_proj(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = scores.softmax(dim=-1)
        y = weights @ v
        y = y.transpose(1, 2).contiguous().view(x.size(0), x.size(1), self.embed_dim)
        return self.out_proj(y), weights

x = torch.randn(2, 5, 32)
mha = MultiHeadAttention(embed_dim=32, num_heads=4)
y, w = mha(x)
print(y.shape, w.shape)  # (2,5,32), (2,4,5,5)


## What to check
**Expected result:** For B=2, T=5, E=32, H=4: weights `(2, 4, 5, 5)` and output `(2, 5, 32)`.

## Pitfalls
Scale by $\sqrt{d_h}$, not $\sqrt{E}$. `.contiguous()` is needed before `.view` after a transpose. Forgetting `transpose(1,2)` attends across heads instead of tokens.